In [ ]:
# Build the climatological high-flow thresholds table,
# iceberg.teehr.nwmd_flow_thresholds.
#
# Split out of nwmd_metrics.ipynb: the thresholds are a property of each gage's
# observed period of record, not of any one metrics run, so they are computed on
# their own schedule and on their own (much smaller) cluster. nwmd_metrics.ipynb
# only reads the table.
import importlib
import pathlib
import sys
import time
import urllib.request

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

In [ ]:
BRANCH = "improve-nwmd-preprocessing-efficiency"
MOD_DIR = pathlib.Path("/home/jovyan/nwmd_modules")
MOD_DIR.mkdir(parents=True, exist_ok=True)

RAW = ("https://raw.githubusercontent.com/RTIInternational/teehr-hub/"
       f"{BRANCH}/warehouse/remote/03_preprocessing/nwm_diagnostics/utils.py")

# cache-buster: raw.githubusercontent caches branch URLs for ~5 minutes
urllib.request.urlretrieve(f"{RAW}?t={time.time()}", MOD_DIR / "utils.py")

sys.path.insert(0, str(MOD_DIR))
import utils
importlib.reload(utils)  # picks up a re-download without a kernel restart
print("loaded", utils.__file__)
print("thresholds ->", utils.THRESHOLD_TABLE, utils.THRESHOLD_QUANTILES)

In [ ]:
# Far smaller than the metrics run's 64 executors. This job is ONE pass over
# primary_timeseries plus one shuffle for the exact percentile -- there is no
# bootstrap, no 4x threshold row expansion and no 3x rollup expansion, which are
# what make the metrics pipeline shuffle-bound. The output is a handful of rows
# per gage.
#
# These numbers are a starting point from that shape, not from a measured run.
# utils.report_utilization() is printed below; if the cluster comes back
# underused, cut executor_instances rather than leaving 16 nodes idle.
#
# Exact `percentile` (not percentile_approx) holds a whole gage's values in one
# task, so keep the 16g heap even though the executor count is low -- a long
# period of record at a busy gage is the memory hazard here, not the row count.
pod_template_path = utils.create_ondemand_pod_template()
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=16,
    executor_memory="16g",
    executor_cores=3,
    pod_template_path=pod_template_path,
    update_configs={
        # 1024, not the metrics run's 4096: ~16x less data reaches the one
        # shuffle here, and oversized partition counts just add task overhead.
        "spark.sql.shuffle.partitions": 1024,
        "spark.kubernetes.executor.annotation.cluster-autoscaler.kubernetes.io/safe-to-evict": "false",
        "spark.executor.memoryOverhead": "8g",
        "spark.executor.processTreeMetrics.enabled": "true",
        # Let a slow jar fetch finish instead of self-exiting at 120s.
        "spark.files.io.connectionTimeout": "600s",
        "spark.network.timeout": "600s",
        "spark.kubernetes.allocation.batch.size": "5",
        "spark.kubernetes.allocation.batch.delay": "10s",
        "spark.io.compression.codec": "zstd",
    }
)

print(spark.sparkContext.uiWebUrl)          # will show :4041, not :4040
print(spark.sparkContext.applicationId)     # should start with "spark-", not "local-"

In [ ]:
# Compute the per-gage climatological high-flow thresholds ONCE, from the
# primary_timeseries period of record, and persist them.
#
# The metrics run reads this table, so the thresholds no longer move when the
# reference_time window moves -- which is what makes a re-run of a single
# quarter comparable with its neighbours. Re-run this only when you
# deliberately want the thresholds to change, e.g. after an observation
# backfill, and expect every downstream metrics row to shift meaning when you
# do. The write is CREATE OR REPLACE TABLE, so a re-run replaces the table
# wholesale rather than merging into it.
#
# Same `finally` discipline as the metrics run: an exception in the build does
# not stop the SparkContext, and executor pods left running overnight are
# expensive. Check for orphans with `kubectl get pods | grep exec` if the kernel
# itself is killed.
run_seconds = None
start = time.perf_counter()
try:
    utils.build_flow_thresholds(spark)
finally:
    run_seconds = time.perf_counter() - start

    # Capture BEFORE stopping -- the Spark REST API stops answering once the
    # session ends, and a crashed run is exactly when the numbers are worth
    # having. Guarded so a failure here cannot prevent the stop below.
    try:
        run_metrics = utils.capture_spark_run_metrics(
            spark, label="nwmd flow thresholds"
        )
        utils.report_utilization(run_metrics, run_seconds)
        failures = utils.get_stage_attempt_failures(spark)
    except Exception as exc:  # noqa: BLE001
        print(f"metrics capture failed: {exc!r}")

    spark.stop()
    print(f"Spark session stopped after {run_seconds:.1f} s.")

In [ ]:
# The build cell stops Spark, so inspection needs a session of its own. A plain
# local session is plenty for reading a few rows per gage.
# SparkSession.getActiveSession() returns None once the session is stopped; note
# `spark._jsc` is NOT a usable check -- it stays a live JavaObject after stop().
from pyspark.sql import SparkSession

if SparkSession.getActiveSession() is None:
    spark = create_spark_session()

spark.sql("USE iceberg.teehr")

In [ ]:
spark.sql(f"""
SELECT *
FROM {utils.THRESHOLD_TABLE}
ORDER BY location_id, quantile
LIMIT 20
""").show(truncate=False)

In [ ]:
# What nwmd_metrics.ipynb will join on. A quantile missing here, or a
# (variable_name, unit_name) pair that does not match the joined timeseries,
# shows up in the metrics run as NULL thresholds -- i.e. no events at all -- so
# it is worth eyeballing before the expensive run.
spark.sql(f"""
SELECT
    variable_name,
    unit_name,
    collect_set(quantile) AS quantiles,
    count(DISTINCT location_id) AS locations,
    min(n_values) AS min_obs_per_location,
    min(por_start) AS por_start,
    max(por_end) AS por_end
FROM {utils.THRESHOLD_TABLE}
GROUP BY variable_name, unit_name
""").show(truncate=False)

In [ ]:
# Safety net. The build cell already stops Spark in its `finally`, so this is
# normally a no-op -- it matters when you started an inspection session above.
try:
    spark.stop()
    print("Spark session stopped.")
except Exception as exc:  # noqa: BLE001
    print(f"spark.stop() failed: {exc!r}")